# TechQuest: 87,425-Movie Semantic Feature Precompute (CUDA / Colab)

This notebook generates the authoritative, one-time 768-dimensional semantic embeddings (`multi-qa-mpnet-base-dot-v1`) and TF-IDF / Count feature matrices for **87,425 canonical movies** from `movie_features.db`.

### Required Build Inputs (No API keys or raw MovieLens CSVs needed):
1. `movie_features.db`
2. `src/` codebase directory (or cloned repo)
3. Optional: `data/processed/checkpoints/` (if resuming from existing compatible checkpoints)

### Expected Outputs:
- `movie_embeddings.npy` (87,425 × 768, float32, L2-normalized, ~268 MB)
- `movie_id_map.json` (TMDB ID to row index mapping)
- `precompute_manifest.json` (provenance, device, SHA256 checksums)
- `count_vectorizer.pkl` & `tfidf_vectorizer.pkl`
- `feature_matrix.pkl`

## 1. Environment Setup & Dependency Installation

In [ ]:
# Install required lightweight dependencies for the precompute build step
!pip install -q sentence-transformers scikit-learn pyyaml numpy

## 2. Hardware & Environment Verification

In [ ]:
import torch
import sentence_transformers
import sys, os

print('=' * 65)
print(' HARDWARE & ENVIRONMENT DIAGNOSTICS')
print('=' * 65)
print(f'PyTorch Version           : {torch.__version__}')
print(f'SentenceTransformers Ver : {sentence_transformers.__version__}')
print(f'CUDA Available            : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_bytes = torch.cuda.get_device_properties(0).total_memory
    vram_gb = vram_bytes / (1024 ** 3)
    print(f'GPU Model                 : {gpu_name}')
    print(f'Total VRAM                : {vram_gb:.2f} GB')
else:
    print('WARNING: CUDA is not available. Please change runtime type to GPU (T4 / V100 / A100).')
print('=' * 65)

## 3. Verify Checkpoint Compatibility (if uploaded)

In [ ]:
from scripts.colab_precompute import step_2_inspect_checkpoints

chk_info = step_2_inspect_checkpoints('data/processed/checkpoints')
print('Checkpoint status:', chk_info)

## 4. Run CUDA Benchmark on 256 Real Movie Texts
Tests batch sizes 128, 256, and 512 to find the fastest, safest batch size for this GPU.

In [ ]:
from src.db.connection import MovieDatabase
from scripts.colab_precompute import step_3_benchmark_cuda

db = MovieDatabase('movie_features.db')
canonical_movies = db.load_canonical_movies()
print(f'Loaded {len(canonical_movies)} canonical movies from database.')
sample_texts = [m['content_text'] for m in canonical_movies]

bench_info = step_3_benchmark_cuda(
    sample_texts=sample_texts,
    test_batch_sizes=[128, 256, 512],
    total_corpus_size=87425
)

## 5. Execute Full Resumable 87,425-Movie Precompute

In [ ]:
from scripts.colab_precompute import step_4_run_precompute

selected_batch_size = bench_info['best_batch_size']
print(f'Running full precompute with measured safe batch size: {selected_batch_size}')

step_4_run_precompute(encode_batch_size=selected_batch_size, chunk_size=2000)

## 6. Validate Artifacts & Packaging

In [ ]:
from scripts.colab_precompute import step_5_validate_artifacts, step_6_package_artifacts

step_5_validate_artifacts('data/processed')
step_6_package_artifacts('data/processed', 'precomputed_artifacts.zip')

# In Google Colab, trigger download automatically:
try:
    from google.colab import files
    files.download('precomputed_artifacts.zip')
except ImportError:
    print('Archive ready at precomputed_artifacts.zip')